(c) Lisa M Dratva, April 2026
# Pan-infection atlas companion website and tutorial notebook

In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
import scanpy as sc

import cell2tcr

ROOT_DIR = Path("/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection")
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

root_dir = str(ROOT_DIR) + os.sep


## Website atlas summary JSON

In [ ]:
# Load the final public atlas object before building the website JSON.
adata = sc.read_h5ad(DATA_DIR / "tcellatlas.h5ad")
adata = adata[adata.obs.pathogen != "Unknown"]


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 5
import json
from pathlib import Path

import pandas as pd


STRING_COLS = [
    "dataset_id",
    "donor_id",
    "sample_id",
    "tissue",
    "disease",
    "pathogen",
    "pathogen_type",
    "exposure",
    "disease_stage",
    "severity",
    "infection_status",
    "modalities",
    "isolation_strategy",
    "annotation_level_1",
    "annotation_level_2",
    "annotation_level_3",
    "annotation_level_4",
    "age",
    "age_decade",
    "doi",
    "author_year_pathogen",
    "vdj_aa",
    "vdj_v",
    "vdj_j",
    "vj_aa",
    "vj_v",
    "vj_j",
    "motif",
]

CROSS_TAB_PAIRS = [
    ("pathogen", "tissue"),
    ("pathogen", "annotation_level_1"),
    ("pathogen", "annotation_level_2"),
    ("pathogen", "annotation_level_3"),
    ("pathogen", "disease"),
    ("pathogen", "disease_stage"),
    ("pathogen", "severity"),
    ("pathogen", "exposure"),
    ("pathogen", "infection_status"),
    ("pathogen", "dataset_id"),
    ("pathogen", "pathogen_type"),
    ("dataset_id", "tissue"),
    ("dataset_id", "annotation_level_1"),
    ("dataset_id", "pathogen"),
    ("disease", "disease_stage"),
    ("tissue", "annotation_level_1"),
    ("annotation_level_1", "annotation_level_2"),
]


def summarize_string_col(s, top_n=30, unknown_label="Unknown"):
    s = pd.Series(s)

    n_null = int(s.isna().sum())
    non_null = s.dropna().astype(str)

    counts = non_null.value_counts()
    n_unique = int(counts.shape[0])

    labels = []
    values = []

    if len(counts) <= top_n:
        shown = counts
        other = None
    else:
        shown = counts.iloc[:top_n]
        other = counts.iloc[top_n:]

    for label, value in shown.items():
        labels.append(str(label))
        values.append(int(value))

    if other is not None and len(other) > 0:
        labels.append(f"Other ({len(other)} more)")
        values.append(int(other.sum()))

    if n_null > 0:
        labels.append(unknown_label)
        values.append(n_null)

    return {
        "type": "categorical",
        "n_unique": n_unique,
        "n_null": n_null,
        "labels": labels,
        "values": values,
    }


def make_crosstab(obs, row_col, col_col, max_rows=30, max_cols=30, unknown_label="Unknown"):
    if row_col not in obs.columns or col_col not in obs.columns:
        return None

    df = obs[[row_col, col_col]].copy()

    df[row_col] = df[row_col].astype("object").where(df[row_col].notna(), unknown_label).astype(str)
    df[col_col] = df[col_col].astype("object").where(df[col_col].notna(), unknown_label).astype(str)

    row_order = df[row_col].value_counts().head(max_rows).index.tolist()
    col_order = df[col_col].value_counts().head(max_cols).index.tolist()

    df = df[df[row_col].isin(row_order) & df[col_col].isin(col_order)]

    tab = pd.crosstab(df[row_col], df[col_col])
    tab = tab.reindex(index=row_order, columns=col_order, fill_value=0)

    return {
        "rows": [str(x) for x in tab.index.tolist()],
        "cols": [str(x) for x in tab.columns.tolist()],
        "data": tab.astype(int).values.tolist(),
    }


def build_atlas_summary(
    obs,
    output_path="atlas_summary.json",
    string_cols=STRING_COLS,
    cross_tab_pairs=CROSS_TAB_PAIRS,
    top_n=30,
):
    obs = obs.copy()

    for col in obs.columns:
        if isinstance(obs[col].dtype, pd.CategoricalDtype):
            obs[col] = obs[col].astype("object")

    summary = {
        "n_total": int(obs.shape[0]),
        "columns": {},
        "cross_tabs": {},
    }

    for col in string_cols:
        if col in obs.columns:
            summary["columns"][col] = summarize_string_col(obs[col], top_n=top_n)
        else:
            print(f"Skipping missing column: {col}")

    for row_col, col_col in cross_tab_pairs:
        tab = make_crosstab(obs, row_col, col_col)
        if tab is not None:
            summary["cross_tabs"][f"{row_col}___{col_col}"] = tab
        else:
            print(f"Skipping missing crosstab: {row_col} x {col_col}")

    output_path = Path(output_path)
    output_path.write_text(
        json.dumps(summary, ensure_ascii=False, separators=(",", ":")),
        encoding="utf-8",
    )

    print(f"Wrote: {output_path}")
    print(f"n_total: {summary['n_total']:,}")
    print(f"columns: {len(summary['columns'])}")
    print(f"cross_tabs: {len(summary['cross_tabs'])}")

    return summary

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 6
summary = build_atlas_summary(
    adata.obs,
    output_path="atlas_summary.json",
)

## Website TCR query table

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 35
import pandas as pd
import cell2tcr
# Cell2TCR motifs
df = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/clone_df_with_motif_21.csv', index_col='Unnamed: 0')
df.head()

# attach BEAM specificities
root_dir = '/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection/'
beam_labels = pd.read_csv(root_dir+'datasets/pan_infection_atlas/miscellaneous/milo_proper/BEAM_pathogen_split_detail.csv', index_col=1)
beam_labels[['beam_hla','beam_epitope','beam_organism']] = beam_labels.virus_detail.str.split('_', expand=True).iloc[:,:3].values
beam_labels


beam_labels = beam_labels[beam_labels.index.isin(df.index)]
df.loc[beam_labels.index, ['beam_hla','beam_epitope','beam_organism']] = beam_labels[['beam_hla','beam_epitope','beam_organism']].values


import numpy as np
# split BEAM VZV-EBV cells 
df.loc[df.study=='GSE275633',['beam_organism','beam_epitope']] = df.loc[df.study=='GSE275633', 'antigen'].str.split('_', expand=True).values
df.loc[df.study=='GSE275633','pathogen'] = df.loc[df.study=='GSE275633', 'antigen'].apply(lambda x: 'EBV_VZV' if isinstance(x, float) else 'EBV' if 'EBV' in x else 'VZV' if 'VZV' in x else np.nan)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 36
# subset of columns for interactive website query 
df[['annotation_level_4', 'annotation_level_3',        'annotation_level_2', 'annotation_level_1', 'annotation_level_0', 'vdj_aa', 'vdj_v', 'vdj_j', 'vj_aa',        'vj_v', 'vj_j', 'clone_id', 'motif', 'beam_hla', 'beam_epitope',        'beam_organism', 'age', 'donor_id', 'sample_id', 'dataset_id', 'pathogen', 'exposure','disease_stage','severity','modalities','tissue','sex','author_year_pathogen']].to_csv('data/clone_df_with_motif_21_w_beam_labels_website.csv')

## cell2specificity tutorial/test data provenance

The public cell2specificity repo already contains tutorial documentation and a bundled toy dataset for tests. This section is kept only as provenance for regenerating that toy data, not as part of the paper figure workflow.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 51
# obs =   # or load from h5ad

# --- Step 1: load your bundled reference motifs to anchor on real motif IDs
motifs_disease = pd.read_csv('data/disease_associated_motifs_hla.csv')
motifs_hla     = pd.read_csv('data/df_motifs_with_hla.csv')
reference_motifs = set(motifs_disease['motif']).union(set(motifs_hla['motif']))

# --- Step 2: require complete paired TCR (both chains)
has_tcr = (
    obs['IR_VDJ_1_junction_aa'].notna() &
    obs['IR_VJ_1_junction_aa'].notna() &
    obs['IR_VDJ_1_v_call'].notna() &
    obs['IR_VJ_1_v_call'].notna()
)

# --- Step 3: require a motif assignment that exists in the reference
has_ref_motif = obs['motif'].isin(reference_motifs)

# --- Step 4: grab MAIT and iNKT cells explicitly (these are rare — take all you have up to a cap)
is_mait = (
    obs['IR_VJ_1_v_call'].str.contains('TRAV1-2', na=False) &
    obs['IR_VJ_1_j_call'].str.contains('TRAJ33|TRAJ20|TRAJ12', na=False, regex=True)
)
is_inkt = (
    obs['IR_VJ_1_v_call'].str.contains('TRAV10', na=False) &
    obs['IR_VJ_1_j_call'].str.contains('TRAJ18', na=False)
)

mait_cells = obs[is_mait & has_tcr].sample(min(50, is_mait.sum()), random_state=42)
inkt_cells = obs[is_inkt & has_tcr].sample(min(20, is_inkt.sum()), random_state=42)

# --- Step 5: pick donors who have ≥1 reference motif (ensures specificity module is testable)
donors_with_ref_motifs = obs[has_tcr & has_ref_motif]['donor_id'].unique()

# --- Step 6: sample conventional T cells — stratified across pathogens, from those donors
conventional = obs[
    has_tcr &
    ~is_mait & ~is_inkt &
    obs['donor_id'].isin(donors_with_ref_motifs)
]

# stratify: up to 60 cells per pathogen × up to 5 donors per pathogen
sampled_conventional = (
    conventional
    .groupby(['pathogen', 'donor_id'], group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 15), random_state=42))
    .groupby('pathogen', group_keys=False)
    .apply(lambda g: g.head(5 * 15))  # cap per pathogen
)

# --- Step 7: combine and keep only the columns the package actually uses
keep_cols = [
    # VDJ (IR_ format — for preprocess_tcr_table)
    'IR_VDJ_1_v_call', 'IR_VDJ_1_j_call', 'IR_VDJ_1_junction_aa',
    'IR_VJ_1_v_call',  'IR_VJ_1_j_call',  'IR_VJ_1_junction_aa',
    # compact format (for matching)
    'vdj_v', 'vdj_j', 'vdj_aa', 'vj_v', 'vj_j', 'vj_aa',
    # metadata
    'donor_id', 'motif', 'pathogen', 'study', 'tissue',
    'clone_id', 'annotation_level_2', 'annotation_level_3',
]

toy = (
    pd.concat([mait_cells, inkt_cells, sampled_conventional])
    .drop_duplicates()
    [[c for c in keep_cols if c in obs.columns]]
    .reset_index(drop=True)
)

print(toy.shape)
print(toy['pathogen'].value_counts())
print(toy[toy['IR_VJ_1_v_call'].str.contains('TRAV1-2', na=False)].shape[0], "MAIT cells")
print(toy['donor_id'].nunique(), "donors")
print(toy['motif'].isin(reference_motifs).sum(), "cells with reference motifs")

toy
# toy.to_csv('tests/data/toy_tcr_atlas.csv')


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 52
toy.to_csv('data/test_dataset.csv')